In [ ]:
from Bio import Entrez, Medline
from groq import Groq
import json
import time
import os
from datetime import datetime
os.environ["GROQ_API_KEY"] = "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"       

client = Groq(api_key=os.environ["GROQ_API_KEY"])


Entrez.email = "kostas.georgiadisg@gmail.com"

In [43]:


# ============================================================
# PRISMA 2020 CRITERIA - RCTs ONLY
# ============================================================
prisma_criteria = {
    "review_info": {
        "title": "AI/ML Interventions for Cancer Diagnosis: A Systematic Review of RCTs",
        "prisma_version": "2020",
        "study_design_focus": "RCTs only"
    },
    
    "pico": {
        "population": {
            "inclusion": [
                "Adults (≥18 years) with cancer or suspected cancer",
                "Adults undergoing cancer screening"
            ],
            "exclusion": [
                "Pediatric populations (<18 years)",
                "Animal studies",
                "Healthy volunteers without cancer risk"
            ]
        },
        "intervention": {
            "inclusion": [
                "AI/ML-based diagnostic tools",
                "AI-assisted clinical decision support systems",
                "ML-based prediction tools used in patient care",
                "Deep learning diagnostic applications"
            ],
            "exclusion": [
                "Traditional statistics without AI/ML",
                "Non-computerized interventions",
                "AI tools not directly used for patient care"
            ]
        },
        "comparator": {
            "inclusion": [
                "Standard care / usual care",
                "Traditional diagnostic methods",
                "Physician judgment alone",
                "Non-AI clinical decision support",
                "Placebo or no intervention"
            ]
        },
        "outcome": {
            "primary": [
                "Diagnostic accuracy (sensitivity, specificity)",
                "Clinical outcomes (survival, recurrence)",
                "Treatment response prediction accuracy"
            ],
            "secondary": [
                "Time to diagnosis",
                "Cost-effectiveness",
                "Patient-reported outcomes",
                "Clinician acceptance/adoption",
                "Adverse events"
            ]
        }
    },
    
    "study_design": {
        "inclusion": [
            "Randomized Controlled Trials (RCTs)",
            "Cluster-randomized trials",
            "Crossover RCTs",
            "Pragmatic RCTs"
        ],
        "exclusion": [
            "Non-randomized studies",
            "Cohort studies",
            "Case-control studies",
            "Cross-sectional studies",
            "Before-after studies without control",
            "Case reports / Case series",
            "Narrative reviews",
            "Systematic reviews / Meta-analyses",
            "Editorials / Commentaries / Letters",
            "Conference abstracts only",
            "Protocols without results",
            "Model development studies without clinical RCT"
        ]
    },
    
    "publication": {
        "language": ["English"],
        "date_range": "2015-01-01 to present",
        "publication_type": ["Peer-reviewed journal articles"]
    },
    
    "quality_assessment": {
        "tool": "Cochrane Risk of Bias 2.0 (RoB 2)",
        "domains": [
            "Randomization process",
            "Deviations from intended interventions",
            "Missing outcome data",
            "Measurement of the outcome",
            "Selection of the reported result"
        ]
    }
}


# ============================================================
# FUNCTIONS
# ============================================================
#Exxpand
def fetch_pubmed(query: str, max_results: int = 10) -> list:
    """
    Fetch papers from PubMed using Biopython.
    Returns structured data matching the standard schema.
    
    Args:
        query: PubMed search query
        max_results: Maximum number of papers to fetch
    
    Returns:
        List of paper dictionaries with full metadata
    """
    print(f"Searching PubMed for: {query[:80]}...")
    
    # Get current timestamp
    retrieved_at = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
    
    # Step 1: Search for PMIDs
    handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results)
    record = Entrez.read(handle)
    handle.close()
    
    pmids = record["IdList"]
    print(f"Found {len(pmids)} PMIDs")
    
    if not pmids:
        return []
    
    # Step 2: Fetch paper details
    handle = Entrez.efetch(db="pubmed", id=pmids, rettype="medline", retmode="text")
    records = Medline.parse(handle)
    
    papers = []
    for rec in records:
        pmid = rec.get("PMID", "")
        
        # Parse authors into structured format
        authors_raw = rec.get("AU", [])
        authors = []
        for author in authors_raw:
            # Author format is usually "LastName FirstInitials" e.g., "Smith JA"
            parts = author.rsplit(" ", 1)
            if len(parts) == 2:
                authors.append({
                    "last_name": parts[0],
                    "initials": parts[1]
                })
            else:
                authors.append({
                    "last_name": author,
                    "initials": ""
                })
        
        # Extract year from date (format: "2023 Jan 15" or "2023")
        date_raw = rec.get("DP", "")
        year = int(date_raw[:4]) if date_raw and date_raw[:4].isdigit() else None
        
        # Build the structured paper object
        paper = {
            # Identifiers
            "record_id": f"PMID:{pmid}",
            "source": "PubMed",
            "retrieved_at": retrieved_at,
            
            # Core content
            "title": rec.get("TI", ""),
            "abstract": rec.get("AB", ""),
            
            # Authors
            "authors": authors,
            
            # Publication info
            "journal": rec.get("JT", ""),
            "year": year,
            "publication_type": rec.get("PT", []),
            "language": rec.get("LA", ["en"])[0] if rec.get("LA") else "en",
            
            # Identifiers
            "doi": rec.get("LID", "").replace(" [doi]", "") if "[doi]" in rec.get("LID", "") else rec.get("AID", [""])[0].replace(" [doi]", "") if rec.get("AID") and "[doi]" in str(rec.get("AID")) else "",
            "pmid": pmid,
            
            # Indexing
            "mesh_terms": rec.get("MH", []),
            "keywords": rec.get("OT", []),  # OT = Other Terms (author keywords)
            
            # URL
            "url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
        }
        
        papers.append(paper)
    
    handle.close()
    return papers


def screen_paper_rct(paper: dict, criteria: dict) -> dict:
    prompt = f"""You are a systematic review screener following PRISMA 2020 guidelines.
IMPORTANT: This review includes ONLY Randomized Controlled Trials (RCTs).

ELIGIBILITY CRITERIA:

POPULATION:
- Include: {', '.join(criteria['pico']['population']['inclusion'])}
- Exclude: {', '.join(criteria['pico']['population']['exclusion'])}

INTERVENTION:
- Include: {', '.join(criteria['pico']['intervention']['inclusion'])}
- Exclude: {', '.join(criteria['pico']['intervention']['exclusion'])}

COMPARATOR:
- Include: {', '.join(criteria['pico']['comparator']['inclusion'])}

OUTCOMES:
- Primary: {', '.join(criteria['pico']['outcome']['primary'])}
- Secondary: {', '.join(criteria['pico']['outcome']['secondary'])}

STUDY DESIGN (CRITICAL - MOST IMPORTANT):
- Include ONLY: {', '.join(criteria['study_design']['inclusion'])}
- Exclude: {', '.join(criteria['study_design']['exclusion'])}

PAPER TO SCREEN:
PMID: {paper['pmid']}
Title: {paper['title']}
Abstract: {paper['abstract'][:2000] if paper['abstract'] else 'No abstract available'}
Publication Type: {', '.join(paper.get('publication_type', [])) if paper.get('publication_type') else 'Not specified'}
Keywords: {', '.join(paper.get('keywords', [])) if paper.get('keywords') else 'None'}

SCREENING TASK:
1. FIRST: Is this an RCT? Look for keywords like: "randomized", "randomly assigned", "random allocation", "randomised", "RCT"
   Also check Publication Type for "Randomized Controlled Trial"
2. If NOT an RCT → decision MUST be "exclude" with exclusion_reason "Not an RCT"
3. If it IS an RCT → evaluate if it meets PICO criteria

IMPORTANT: Set confidence between 0.7 and 0.99 based on how clear the decision is.
- 0.90-0.99 = Very clear case (obvious RCT or obviously not RCT)
- 0.70-0.89 = Fairly confident
- 0.50-0.69 = Uncertain, needs manual review

Respond with ONLY valid JSON (no markdown, no extra text):
{{
    "decision": "include",
    "confidence": 0.85,
    "is_rct": true,
    "population_met": true,
    "intervention_met": true,
    "comparator_met": true,
    "outcome_met": true,
    "exclusion_reason": null,
    "reasoning": "Brief 1-2 sentence explanation"
}}
"""
    
    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            max_tokens=400
        )
        
        text = response.choices[0].message.content.strip()
        
        # Clean markdown if present
        if "```" in text:
            text = text.split("```")[1]
            if text.startswith("json"):
                text = text[4:]
            text = text.strip()
        
        result = json.loads(text)
        
        # FIX: Αν το confidence είναι 0 αλλά έχει valid decision, βάλε default
        if result.get('confidence') == 0.0 and result.get('decision') in ['include', 'exclude']:
            if result.get('is_rct') == False:
                result['confidence'] = 0.90  # Confident exclude (not RCT)
            elif result.get('is_rct') == True:
                result['confidence'] = 0.85  # Confident include
            else:
                result['confidence'] = 0.50  # Uncertain
        
        # Safety check: if is_rct is False, force exclude
        if result.get('is_rct') == False and result.get('decision') != 'exclude':
            result['decision'] = 'exclude'
            result['exclusion_reason'] = 'Not an RCT (auto-corrected)'
        
        return result
        
    except json.JSONDecodeError as e:
        return {
            "decision": "uncertain",
            "confidence": 0.0,
            "is_rct": None,
            "population_met": None,
            "intervention_met": None,
            "comparator_met": None,
            "outcome_met": None,
            "exclusion_reason": f"JSON parse error: {str(e)[:30]}",
            "reasoning": f"Raw response: {text[:100] if 'text' in dir() else 'No response'}"
        }
    except Exception as e:
        return {
            "decision": "uncertain",
            "confidence": 0.0,
            "is_rct": None,
            "population_met": None,
            "intervention_met": None,
            "comparator_met": None,
            "outcome_met": None,
            "exclusion_reason": f"Error: {str(e)[:50]}",
            "reasoning": "API error"
        }

# ============================================================
# MAIN EXECUTION
# ============================================================

# 1. FETCH PAPERS
print("=" * 60)
print("STEP 1: FETCHING PAPERS FROM PUBMED")
print("=" * 60)

#query = "(Cancer OR cancer) AND (randomized controlled trial OR randomised OR clinical trial)"
query = """
(("artificial intelligence" OR "machine learning" OR "deep learning" 
  OR "neural network" OR "computer-aided" OR "AI-assisted") 
 AND (cancer OR neoplasm OR tumor OR tumour OR carcinoma OR oncology) 
 AND (diagnosis OR detection OR screening)
 AND (randomized controlled trial[pt] OR randomized[tiab] OR randomised[tiab]))
"""
papers = fetch_pubmed(query, max_results=1000)
print(f"\nFetched {len(papers)} papers\n")


# 2. DISPLAY FETCHED PAPERS (with new structure)
print("=" * 60)
print("STEP 2: FETCHED PAPERS")
print("=" * 60)

for i, p in enumerate(papers):
    print(f"\n{i+1}. [{p['record_id']}]")
    print(f"   Title: {p['title'][:60]}...")
    print(f"   Year: {p['year']} | Journal: {p['journal'][:30]}")
    print(f"   Pub Type: {', '.join(p['publication_type'][:2]) if p['publication_type'] else 'N/A'}")
    print(f"   DOI: {p['doi'][:30] if p['doi'] else 'N/A'}")
    print(f"   URL: {p['url']}")


# 3. SCREEN PAPERS
print("\n" + "=" * 60)
print("STEP 3: AI SCREENING (PRISMA 2020 - RCTs ONLY)")
print("=" * 60)

screening_results = []

for i, paper in enumerate(papers):
    print(f"\n[{i+1}/{len(papers)}] Screening: {paper['title'][:50]}...")
    
    # Screen the paper
    decision = screen_paper_rct(paper, prisma_criteria)
    
    # Add screening results to paper
    paper_with_screening = {
        **paper,
        "screening": {
            "decision": decision['decision'],
            "confidence": decision['confidence'],
            "is_rct": decision.get('is_rct'),
            "pico_assessment": {
                "population_met": decision.get('population_met'),
                "intervention_met": decision.get('intervention_met'),
                "comparator_met": decision.get('comparator_met'),
                "outcome_met": decision.get('outcome_met')
            },
            "exclusion_reason": decision.get('exclusion_reason'),
            "reasoning": decision.get('reasoning'),
            "screened_at": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
        }
    }
    screening_results.append(paper_with_screening)
    
    # Display result
    is_rct = decision.get('is_rct')
    rct_status = "✓ RCT" if is_rct == True else "✗ Not RCT" if is_rct == False else "? Unknown"
    
    emoji = "✅" if decision['decision'] == 'include' else "❌" if decision['decision'] == 'exclude' else "❓"
    
    print(f"   {emoji} {decision['decision'].upper()} | {rct_status} | Confidence: {decision['confidence']}")
    print(f"   Reasoning: {decision.get('reasoning', 'N/A')}")
    
    if decision.get('exclusion_reason'):
        print(f"   Exclusion reason: {decision['exclusion_reason']}")
    
    # Rate limiting
    time.sleep(3)


# 4. SUMMARY
print("\n" + "=" * 60)
print("STEP 4: SCREENING RESULTS SUMMARY")
print("=" * 60)

included = [r for r in screening_results if r['screening']['decision'] == 'include']
excluded = [r for r in screening_results if r['screening']['decision'] == 'exclude']
uncertain = [r for r in screening_results if r['screening']['decision'] == 'uncertain']

non_rct = [r for r in excluded if r['screening'].get('is_rct') == False]
other_exclusions = [r for r in excluded if r['screening'].get('is_rct') != False]

print(f"\n✅ INCLUDE:          {len(included)} papers")
print(f"❌ EXCLUDE:          {len(excluded)} papers")
print(f"   └─ Not RCT:       {len(non_rct)}")
print(f"   └─ Other reasons: {len(other_exclusions)}")
print(f"❓ UNCERTAIN:        {len(uncertain)} papers")
print(f"{'─'*45}")
print(f"   TOTAL SCREENED:   {len(papers)} papers")


# 5. SHOW INCLUDED PAPERS
if included:
    print("\n" + "=" * 60)
    print("INCLUDED RCTs (for full-text review)")
    print("=" * 60)
    for i, p in enumerate(included):
        print(f"\n{i+1}. [{p['record_id']}]")
        print(f"   Title: {p['title']}")
        print(f"   Year: {p['year']} | Journal: {p['journal'][:40]}")
        print(f"   DOI: {p['doi'] if p['doi'] else 'N/A'}")
        print(f"   URL: {p['url']}")
        print(f"   Confidence: {p['screening']['confidence']}")
        print(f"   Reasoning: {p['screening']['reasoning']}")


# 6. SHOW EXCLUDED PAPERS
if excluded:
    print("\n" + "=" * 60)
    print("EXCLUDED PAPERS")
    print("=" * 60)
    for i, p in enumerate(excluded[:10]):  # Show first 10
        print(f"\n{i+1}. [{p['record_id']}] {p['title'][:50]}...")
        print(f"   Reason: {p['screening'].get('exclusion_reason', 'Not specified')}")


# 7. SAVE RESULTS
# All results
with open("rct_screening_results.json", "w", encoding="utf-8") as f:
    json.dump(screening_results, f, indent=2, ensure_ascii=False)

# Only included
with open("rct_included_papers.json", "w", encoding="utf-8") as f:
    json.dump(included, f, indent=2, ensure_ascii=False)

# Only excluded
with open("rct_excluded_papers.json", "w", encoding="utf-8") as f:
    json.dump(excluded, f, indent=2, ensure_ascii=False)

# PRISMA flow data
prisma_flow = {
    "review_info": prisma_criteria["review_info"],
    "search_info": {
        "database": "PubMed",
        "query": query,
        "date_searched": datetime.utcnow().strftime("%Y-%m-%d"),
        "total_results": len(papers)
    },
    "identification": {
        "records_identified": len(papers),
        "duplicates_removed": 0
    },
    "screening": {
        "records_screened": len(papers),
        "records_excluded": len(excluded),
        "exclusion_reasons": {
            "not_rct": len(non_rct),
            "pico_not_met": len(other_exclusions),
            "uncertain": len(uncertain)
        }
    },
    "eligibility": {
        "full_text_assessed": len(included),
        "full_text_excluded": 0
    },
    "included": {
        "studies_in_review": len(included)
    }
}

with open("prisma_flow_data.json", "w", encoding="utf-8") as f:
    json.dump(prisma_flow, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 60)
print("FILES SAVED")
print("=" * 60)
print("📄 rct_screening_results.json  - All papers with full metadata + screening")
print("📄 rct_included_papers.json    - Included RCTs only")
print("📄 rct_excluded_papers.json    - Excluded papers with reasons")
print("📄 prisma_flow_data.json       - Data for PRISMA flow diagram")
print("\n✨ Done!")

STEP 1: FETCHING PAPERS FROM PUBMED
Searching PubMed for: 
(("artificial intelligence" OR "machine learning" OR "deep learning" 
  OR "neu...
Found 1000 PMIDs


KeyboardInterrupt: 

AttributeError: partially initialized module 'charset_normalizer' has no attribute 'md__mypyc' (most likely due to a circular import)